# Flood Wall Simulations — Lao PDR

This notebook simulates the effect of building floodwalls in five Laotian cities by increasing the effective distance to rivers in the geospatial damage model.

**Three scenarios:**
1. **No wall (baseline):** Original distance-to-river values
2. **Intermediate wall:** Distance-to-river set to the **median** value across the Laos prediction grid
3. **Full wall:** Distance-to-river set to the **maximum** value across the Laos prediction grid

**Cities:** Luang Prabang, Vientiane, Savannakhet, Pakse, Attapeu

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
from pyprojroot import here
sys.path.insert(0, str(here()))
from os.path import exists

import pandas as pd
import numpy as np
import pymc as pm
import pytensor
import pytensor.tensor as pt
import matplotlib.pyplot as plt
import arviz as az
import xarray as xr
import pathlib

from laos_gggi.model import add_data
from laos_gggi.plotting import configure_plot_style
from laos_gggi.data_functions import load_shapefile
from sklearn.preprocessing import StandardScaler as Standardize
from pymc.model.transform.optimization import freeze_dims_and_data
from scipy import interpolate
from cycler import cycler

In [3]:
# Paths
GEO_MODEL_DIR = here('notebooks/geo_damage_model')
PRETRAINED_IDATA_DIR = here('notebooks/final_paper/idata')
IDATA_DIR = here('notebooks/policy_note/idata')
FIGURES_DIR = here('notebooks/policy_note/figures')

# Style
OECD_colors = ["#002F6C", "#7FA8D9", "#006BB6", "#00AACC", "#82D2E3"]
OECD_cycler = cycler(color=OECD_colors)

config = {
    'figure.figsize': (14, 4),
    'figure.constrained_layout.use': True,
    'figure.facecolor': 'w',
    'axes.grid': False,
    'grid.linewidth': 0.5,
    'grid.linestyle': '--',
    'axes.spines.top': False,
    'axes.spines.bottom': False,
    'axes.spines.left': False,
    'axes.spines.right': False,
    'axes.prop_cycle': OECD_cycler,
}
plt.rcParams.update(config)

# Constants
WALL_CITIES = ["Luang Prabang", "Vientiane", "Savannakhet", "Pakse", "Attapeu"]
PREDICTION_YEARS = ["2026-01-01", "2030-01-01", "2040-01-01", "2050-01-01", "2070-01-01"]
SCENARIO_COLORS = {"No wall": "#002F6C", "Intermediate wall": "#006BB6", "Full wall": "#82D2E3"}
SEED = sum(list(map(ord, 'climate_bayes')))
rng = np.random.default_rng(SEED)
floatX = pytensor.config.floatX

# Section 1: Load Pre-trained Posteriors

In [4]:
# SEA countries used for training
laos_neighboors = ["KHM", "THA", "LAO", "VNM"]

# Load pre-trained model idatas from final_paper/idata/
# Event model (HSGP geospatial): generated by 05b_geo_model_laos_neigh.ipynb
event_idata = az.from_netcdf(PRETRAINED_IDATA_DIR / "sea_geo_model.idata")

# Damage model: generated by damage modeling pipeline
damage_idata = az.from_netcdf(PRETRAINED_IDATA_DIR / "dmg_5_2025_usd_new_b.nc")

# Rename to avoid conflicts when merging (same as 04_b_damage_curves_lao.ipynb)
damage_idata = damage_idata.rename({
    "country_loc": "country_loc_damage",
    "country_scale": "country_scale_damage",
    "country_offset": "country_offset_damage",
    "country_effect": "country_effect_damage",
    "beta": "beta_damage",
    "beta_GY": "beta_GY_damage",
    "mixture": "mixture_damage",
    "alpha_X": "alpha_X_damage",
    "latent_iso": "latent_iso_damage",
    "phi": "phi_damage",
    "phi_climatological_disasters": "phi_climatological_disasters_damage",
    "phi_hydrological_disasters": "phi_hydrological_disasters_damage",
    "rho": "rho_damage",
    "sigma_X": "sigma_X_damage",
    "theta": "theta_Damage",
    "p_zero": "p_zero_damage",
    "mu": "mu_damage",
})

# Merge posteriors (event + damage)
merged_posteriors = xr.merge([
    damage_idata.posterior.drop_vars(['mu_damage']).drop_dims('obs_idx'),
    event_idata.sel(ISO=laos_neighboors).posterior,
])

# Section 2: Load & Prepare Prediction Grid

In [5]:
# Grid columns to load
grid_cols = ['long', 'lat', 'distance_to_river', 'log_distance_to_river',
             'log_distance_to_coastline', 'geometry', 'ISO',
             'log_distance_to_river__standardized', 'log_distance_to_coastline__standardized',
             'log_distance_to_river__standardized__squared',
             'log_distance_to_coastline__standardized__squared']

# Load datasets
sea_point_grid = pd.read_csv(here("data/sea_point_grid.csv"), index_col=0)[grid_cols]
laos_point_grid = pd.read_csv(here("data/laos_point_grid.csv"), index_col=0)[grid_cols]
sea_df = pd.read_csv(here("data/sea.csv"), index_col=0)
sea_df_stand = pd.read_csv(here("data/sea_df_stand.csv"), index_col=0)
damage_df_stand = pd.read_csv(here("data/damage_df_stand.csv"), index_col=0)
damage_df_stand["Population"] = damage_df_stand["Population"] * 1e6

# Load predictions
predictions = pd.read_csv(here("data/climate_forecast.csv")).rename(columns={"time": "Start_Year"})
predictions["ISO"] = "LAO"

# Select years
predictions_short = predictions.query('Start_Year in @PREDICTION_YEARS')

# Create grid point number and merge
laos_point_grid = laos_point_grid.reset_index().rename(columns={"index": "point_number"})
pred_df = pd.merge(laos_point_grid, predictions_short, left_on="ISO", right_on="ISO", how="left")

# Derived features
pred_df["population_density"] = pred_df["Population"] / 236800
pred_df["log_population_density"] = np.log(pred_df["population_density"])
pred_df["log_gdp_per_cap"] = np.log(pred_df["gdp_per_cap"])

# Load city locations
cities_loc = pd.read_csv(here("data/cites_loc.csv"), index_col=0)
print(f"Cities: {list(cities_loc.index)}")
cities_loc

KeyError: "['log_distance_to_river__standardized__squared', 'log_distance_to_coastline__standardized__squared'] not in index"

In [ ]:
# === Damage features (country-level, matching dmg_5_2025_usd_new_b.nc) ===
# These feature names must match the `features_damages` dim in the damage posterior
damage_features_base = ['ln_population_density', 'ln_gdp_pc', 'precip_deviation', 'co2', 'population']
damage_features_base_d = ['d_' + x for x in damage_features_base]
damage_features_d_stand = [x + "__standardized" for x in damage_features_base_d]

# Map geo prediction columns to country-level feature names
# The underlying data is the same, just named differently
pred_df["d_ln_population_density"] = np.log(pred_df["population_density"])
pred_df["d_ln_gdp_pc"] = np.log(pred_df["gdp_per_cap"])
pred_df["d_precip_deviation"] = pred_df["precip_deviation"]
pred_df["d_co2"] = pred_df["co2"]
pred_df["d_population"] = pred_df["Population"]

# Standardize event features (fitted on SEA training data)
cols_to_stand = ['Population', 'dev_ocean_temp', 'co2', 'log_population_density',
                 'log_gdp_per_cap', 'precip_deviation']
cols_to_stand_stand = [x + "__standardized" for x in cols_to_stand]
cols_not_stand = ['ISO', 'Start_Year', 'lat', 'long', 'geometry', 'point_number',
                  'log_distance_to_river__standardized', 'log_distance_to_coastline__standardized',
                  'log_distance_to_river__standardized__squared', 'log_distance_to_coastline__standardized__squared']

transformer_stand_ = Standardize().fit(sea_df[cols_to_stand])
pred_df_stand = transformer_stand_.transform(pred_df[cols_to_stand])
pred_df_stand = pd.DataFrame(pred_df_stand, columns=cols_to_stand_stand)
pred_df_stand = pd.merge(pred_df_stand, pred_df[cols_not_stand], left_index=True, right_index=True, how="left")

# Standardize damage features (fitted on damage training data)
transformer_stand__damage = Standardize().fit(damage_df_stand[damage_features_base_d])
pred_df_stand_damage = transformer_stand__damage.transform(pred_df[damage_features_base_d])
pred_df_stand_damage = pd.DataFrame(pred_df_stand_damage, columns=damage_features_d_stand)
pred_df_stand = pd.merge(pred_df_stand, pred_df_stand_damage, left_index=True, right_index=True, how="left")

# Center lat/long using SEA bounds
sea_center = {}
for x in ["lat", "long"]:
    sea_center[x] = (sea_df[x].max() + sea_df[x].min()) / 2
    pred_df_stand[x + "_centered"] = pred_df_stand[x] - sea_center[x]

# Split into per-year DataFrames
pred_df_stand_dict = {}
for year in pred_df_stand["Start_Year"].unique():
    pred_df_stand_dict[year] = pred_df_stand.query('Start_Year == @year').copy()

print(f"Years: {list(pred_df_stand_dict.keys())}")
print(f"Grid points per year: {len(pred_df_stand_dict[PREDICTION_YEARS[0]])}")

In [ ]:
# Extract coordinate values from merged posteriors to ensure exact match
ISO_coords = list(merged_posteriors.coords['ISO'].values)
iso_coords = list(merged_posteriors.coords['iso'].values)
disaster_coords = list(merged_posteriors.coords['disaster'].values)
feature_coords = list(merged_posteriors.coords['feature'].values)
features_damages_coords = list(merged_posteriors.coords['features_damages'].values)
gp_feature_coords = list(merged_posteriors.coords['gp_feature'].values)

# Event features: 8 features matching 'feature' dim in event posterior
event_features = list(feature_coords)

# Damage features: matching 'features_damages' dim in damage posterior
damage_features = list(features_damages_coords)

# Verify damage feature columns exist in prediction DataFrame
ref_df_check = pred_df_stand_dict[PREDICTION_YEARS[0]]
for f in damage_features:
    assert f in ref_df_check.columns, f"Missing damage feature column: {f}"
print(f"All damage feature columns verified in prediction DataFrame.")

# GP features
gp_features = ["lat", "long"]

# Index lookups
ISO_to_idx = {name: idx for idx, name in enumerate(ISO_coords)}
iso_to_idx = {name: idx for idx, name in enumerate(iso_coords)}
disaster_to_idx = {name: idx for idx, name in enumerate(disaster_coords)}

ISO_idx_laos = ISO_to_idx['LAO']
iso_idx_laos = iso_to_idx['LAO']

# Determine hydro disaster index
hydro_key = [k for k in disaster_to_idx if 'hydro' in k.lower()]
assert len(hydro_key) == 1, f"Expected one hydro disaster, found: {hydro_key}"
hydro_idx = disaster_to_idx[hydro_key[0]]

# Obs idx and years
obs_idx = np.arange(len(pred_df_stand_dict[PREDICTION_YEARS[0]]))
years = pred_df_stand["Start_Year"].unique()

coords_predictions = {
    "obs_idx": obs_idx,
    "ISO": ISO_coords,
    "iso": iso_coords,
    "disaster": disaster_coords,
    "feature": feature_coords,
    "features_damages": features_damages_coords,
    "gp_feature": gp_feature_coords,
}

print(f"\nEvent features ({len(event_features)}): {event_features}")
print(f"Damage features ({len(damage_features)}): {damage_features}")
print(f"Disaster classes: {disaster_coords}")
print(f"LAO index in ISO (event): {ISO_idx_laos}")
print(f"LAO index in iso (damage): {iso_idx_laos}")
print(f"Hydro disaster index: {hydro_idx} ({hydro_key[0]})")

# Section 3: Locate Cities & Compute Wall Distance Values

In [ ]:
# Find nearest grid points for each city
ref_df = pred_df_stand_dict[PREDICTION_YEARS[0]]

best_loc = {}
for city in WALL_CITIES:
    city_lat = cities_loc.loc[city, "lat"]
    city_long = cities_loc.loc[city, "long"]
    dist_sq = (ref_df["lat"] - city_lat) ** 2 + (ref_df["long"] - city_long) ** 2
    best_idx = dist_sq.idxmin()
    best_loc[city] = ref_df.loc[best_idx, ["lat", "long", "point_number"]]
    print(f"{city}: point_number={int(best_loc[city]['point_number'])}, "
          f"lat={best_loc[city]['lat']:.4f}, long={best_loc[city]['long']:.4f}")

# Compute max and median river distance values from Laos grid
max_river_0 = ref_df["log_distance_to_river__standardized"].max()
max_river_1 = ref_df["log_distance_to_river__standardized__squared"].max()
median_river_0 = ref_df["log_distance_to_river__standardized"].median()
median_river_1 = ref_df["log_distance_to_river__standardized__squared"].median()

print(f"\nMax river distance (standardized): {max_river_0:.4f}")
print(f"Max river distance squared (standardized): {max_river_1:.4f}")
print(f"Median river distance (standardized): {median_river_0:.4f}")
print(f"Median river distance squared (standardized): {median_river_1:.4f}")

In [ ]:
# Create wall columns for both scenarios (full and intermediate)
river_variables = ["log_distance_to_river__standardized", "log_distance_to_river__standardized__squared"]
wall_full_variables = [x + "_wall_full" for x in river_variables]
wall_median_variables = [x + "_wall_median" for x in river_variables]

index_val = {}

for city in WALL_CITIES:
    index_val[city] = {}
    for year in years:
        df_year = pred_df_stand_dict[year]

        # Initialize wall columns from originals
        df_year[wall_full_variables[0]] = df_year["log_distance_to_river__standardized"].copy()
        df_year[wall_full_variables[1]] = df_year["log_distance_to_river__standardized__squared"].copy()
        df_year[wall_median_variables[0]] = df_year["log_distance_to_river__standardized"].copy()
        df_year[wall_median_variables[1]] = df_year["log_distance_to_river__standardized__squared"].copy()

        # Find city index
        point_num = best_loc[city].loc["point_number"]
        city_idx = df_year.query('point_number == @point_num').index.values[0]
        index_val[city][year] = city_idx

        # Full wall: set to max
        df_year.loc[city_idx, wall_full_variables[0]] = max_river_0
        df_year.loc[city_idx, wall_full_variables[1]] = max_river_1

        # Intermediate wall: set to median
        df_year.loc[city_idx, wall_median_variables[0]] = median_river_0
        df_year.loc[city_idx, wall_median_variables[1]] = median_river_1

# Wall feature lists: replace index 0 (log_distance_to_river__standardized)
# in the 8-feature event model
river_feat_idx = event_features.index("log_distance_to_river__standardized")

event_features_wall_full = event_features.copy()
event_features_wall_full[river_feat_idx] = wall_full_variables[0]

event_features_wall_median = event_features.copy()
event_features_wall_median[river_feat_idx] = wall_median_variables[0]

print("Wall feature lists defined.")
print(f"River feature at index {river_feat_idx}")
print(f"Full wall replaces: {event_features[river_feat_idx]} -> {event_features_wall_full[river_feat_idx]}")
print(f"Median wall replaces: {event_features[river_feat_idx]} -> {event_features_wall_median[river_feat_idx]}")

# Section 4: Build & Sample Models

For each scenario (base, full wall, intermediate wall), we build the PyMC model and sample posterior predictive distributions. Results are cached to disk.

In [ ]:
def build_damage_curve_model(year, event_feat_list, coords_predictions,
                             pred_df_stand_dict, damage_features,
                             ISO_idx_laos, iso_idx_laos, hydro_idx):
    """Build the combined HSGP event + country-level damage PyMC model.

    Event model: HSGP geospatial (from sea_geo_model.idata)
    Damage model: HurdleLogNormal country-level (from dmg_5_2025_usd_new_b.nc)
    Combined: damages_curves = y_hat_damages * event_prob_y_hat
    """
    df_year = pred_df_stand_dict[year]
    n_obs = len(df_year)

    model = pm.Model(coords=coords_predictions)
    with model:
        # ==================== Event Model (HSGP geospatial) ====================
        # Features: 8 standardized features matching 'feature' dim
        X = add_data(
            features=event_feat_list, target=None,
            df=df_year, dims=['obs_idx', 'feature']
        )
        X_gp = pm.Data(
            "X_gp",
            df_year[["lat_centered", "long_centered"]].astype(floatX),
            dims=['obs_idx', 'gp_feature']
        )

        # Country effect (matches add_country_effect() in model.py)
        country_effect_mu = pm.Flat("country_effect_mu")
        country_effect_scale = pm.Flat("country_effect_scale")
        country_effect_offset = pm.Flat("country_effect_offset", dims=["ISO"])
        country_effect = pm.Deterministic(
            "country_effect",
            country_effect_mu + country_effect_scale * country_effect_offset,
            dims=["ISO"]
        )

        beta = pm.Flat("beta", dims=["feature"])

        # HSGP (Matern 5/2, m=[35,35], c=1.5)
        eta = pm.Flat("eta")
        ell = pm.Flat("ell", dims=["gp_feature"])
        cov_func = eta**2 * pm.gp.cov.Matern52(input_dim=2, ls=ell)
        gp = pm.gp.HSGP(m=[35, 35], c=1.5, cov_func=cov_func)
        gp._X_center = compute_center(
            df_year[["lat_centered", "long_centered"]].values.astype(floatX)
        )
        phi, sqrt_psd = gp.prior_linearized(X=X_gp)
        basis_coeffs = pm.Flat("basis_coeffs", size=gp.n_basis_vectors)
        HSGP_component = pm.Deterministic(
            'HSGP_component', phi @ (basis_coeffs * sqrt_psd), dims=['obs_idx']
        )

        event_features_component = pm.Deterministic(
            'event_features_component', X @ beta, dims=['obs_idx']
        )
        logit_p = pm.Deterministic(
            'logit_p',
            country_effect[ISO_idx_laos] + event_features_component + HSGP_component,
            dims=['obs_idx']
        )
        event_prob_y_hat = pm.Deterministic(
            'event_prob_y_hat', pm.math.invlogit(logit_p), dims=['obs_idx']
        )

        # ==================== Damage Model (HurdleLogNormal) ====================
        # Country-level features matching 'features_damages' dim
        X_damage = pm.Data(
            'X_damage', df_year[damage_features].astype(floatX),
            dims=['obs_idx', 'features_damages']
        )

        # ISO index for damage model (all LAO)
        iso_idx_pt = pm.Data(
            'iso_idx', np.full(n_obs, iso_idx_laos, dtype=int),
            dims=['obs_idx']
        )

        # Latent ISO effect on features
        latent_iso_damage = pm.Flat('latent_iso_damage', dims=['iso', 'features_damages'])

        # Hierarchical country effect for damages
        country_loc_damage = pm.Flat('country_loc_damage', dims=['disaster'])
        country_scale_damage = pm.Flat('country_scale_damage', dims=['disaster'])
        country_offset_damage = pm.Flat('country_offset_damage', dims=['iso', 'disaster'])
        country_effect_damage = pm.Deterministic(
            'country_effect_damage',
            country_loc_damage + country_scale_damage * country_offset_damage,
            dims=['iso', 'disaster']
        )

        beta_damage = pm.Flat('beta_damage', dims=['features_damages', 'disaster'])
        beta_GY_damage = pm.Flat('beta_GY_damage', dims=['features_damages', 'disaster'])
        coef_effect_damage = X_damage @ beta_damage + (latent_iso_damage @ beta_GY_damage)[iso_idx_pt]

        mixture_damage = pm.Flat("mixture_damage", dims=["iso", 'disaster'])

        mu_full = (country_effect_damage + mixture_damage)[iso_idx_pt] + coef_effect_damage

        sigma = pm.Flat('sigma', dims=['disaster'])

        # All grid points are hydro disasters — select hydro column directly
        mu_damage = pm.Deterministic('mu_damage', mu_full[:, hydro_idx], dims=['obs_idx'])

        p_zero_damage = pm.Flat('p_zero_damage', dims=['disaster'])
        y_hat_damages = pm.HurdleLogNormal(
            'y_hat_damages',
            mu=mu_damage,
            sigma=sigma[hydro_idx],
            psi=(1 - p_zero_damage)[hydro_idx],
            observed=np.zeros(n_obs),
            dims=['obs_idx']
        )

        # ==================== Combined Damage Curves ====================
        damages_curves = pm.Deterministic(
            "damages_curves", y_hat_damages * event_prob_y_hat, dims=['obs_idx']
        )

    return model

print("Model builder defined.")

In [ ]:
def sample_scenario(scenario_name, event_feat_list, cache_prefix):
    """Build models and sample posterior predictive for all years, with caching."""
    idata_dict = {}
    for year in years:
        cache_path = IDATA_DIR / f"{cache_prefix}_{year}.idata"
        if exists(cache_path):
            print(f"  Loading cached {scenario_name} {year}...")
            idata_dict[year] = az.from_netcdf(cache_path)
        else:
            print(f"  Sampling {scenario_name} {year}...")
            model = build_damage_curve_model(
                year, event_feat_list, coords_predictions,
                pred_df_stand_dict, damage_features,
                ISO_idx_laos, iso_idx_laos, hydro_idx
            )
            with freeze_dims_and_data(model):
                idata_dict[year] = pm.sample_posterior_predictive(
                    merged_posteriors, extend_inferencedata=False,
                    compile_kwargs={'mode': 'JAX'},
                    var_names=['damages_curves', 'y_hat_damages',
                               'event_prob_y_hat', 'mu_damage']
                )
            az.to_netcdf(data=idata_dict[year], filename=pathlib.Path(cache_path))
    return idata_dict

In [ ]:
# Sample all three scenarios
print("=== Base scenario (no wall) ===")
idata_base = sample_scenario("Base", event_features, "damage_curves_base")

print("\n=== Full wall scenario (max distance) ===")
idata_wall_full = sample_scenario("Full wall", event_features_wall_full, "damage_curves_walls_full")

print("\n=== Intermediate wall scenario (median distance) ===")
idata_wall_median = sample_scenario("Intermediate wall", event_features_wall_median, "damage_curves_walls_median")

print("\nAll scenarios sampled.")

# Section 5: Extract City Posteriors & Compute Return Years

In [ ]:
# Build mapping from point_number to positional obs_idx for each year
point_to_obs_idx = {}
for year in years:
    df_year = pred_df_stand_dict[year]
    point_to_obs_idx[year] = {
        int(pn): pos for pos, pn in enumerate(df_year["point_number"].values)
    }

# Extract city-level posteriors for each scenario
scenario_idatas = {
    "No wall": idata_base,
    "Intermediate wall": idata_wall_median,
    "Full wall": idata_wall_full,
}

city_posteriors = {}  # {city: {scenario: {year: array}}}

for city in WALL_CITIES:
    city_posteriors[city] = {}
    point_num = int(best_loc[city]["point_number"])
    for scenario_name, idata_dict in scenario_idatas.items():
        city_posteriors[city][scenario_name] = {}
        for year in years:
            obs_pos = point_to_obs_idx[year][point_num]
            city_idata = idata_dict[year].sel(obs_idx=obs_pos)
            damages = city_idata.posterior_predictive["damages_curves"].values.ravel()
            city_posteriors[city][scenario_name][year] = damages

print(f"Extracted posteriors for {len(WALL_CITIES)} cities x {len(scenario_idatas)} scenarios x {len(years)} years")

In [ ]:
def compute_return_year_damages(posterior_samples, return_years=[25, 30, 40, 50]):
    """Compute damages at given return year periods from posterior samples."""
    data = np.sort(posterior_samples)
    n = len(data)
    cdf_vals = np.linspace(1/n, 1, n)
    ppf_func = interpolate.interp1d(cdf_vals, data, bounds_error=False,
                                     fill_value=(data[0], data[-1]))
    results = {}
    for ry in return_years:
        results[f"{ry} RY"] = float(ppf_func(1 - 1/ry))
    return results

# Compute return year damages for all cities, scenarios, years
RETURN_YEARS = [25, 30, 40, 50]

ry_results = []
for city in WALL_CITIES:
    for scenario_name in scenario_idatas.keys():
        for year in years:
            ry_damages = compute_return_year_damages(
                city_posteriors[city][scenario_name][year], RETURN_YEARS
            )
            for ry_label, damage_val in ry_damages.items():
                ry_results.append({
                    "City": city,
                    "Scenario": scenario_name,
                    "Year": year[:4],
                    "Return Period": ry_label,
                    "Damage (USD millions)": damage_val,
                })

ry_df = pd.DataFrame(ry_results)
print(f"Return year table: {ry_df.shape[0]} rows")
ry_df.head(12)

# Section 6: Posterior Comparison Plots

In [ ]:
# Posterior density comparison: per city, base vs full wall
years_to_plot = ['2026-01-01', '2030-01-01', '2050-01-01']
scenarios_to_plot = ["No wall", "Full wall"]

for city in WALL_CITIES:
    n = len(years_to_plot)
    fig, axes = plt.subplots(n, 2, figsize=(12, 2.5 * n), sharex=True)
    point_num = int(best_loc[city]["point_number"])

    for col, scenario_name in enumerate(scenarios_to_plot):
        idata_dict = scenario_idatas[scenario_name]
        for row, year in enumerate(years_to_plot):
            ax = axes[row, col]
            obs_pos = point_to_obs_idx[year][point_num]
            city_idata = idata_dict[year].sel(obs_idx=obs_pos)
            az.plot_posterior(
                city_idata.posterior_predictive,
                var_names=["damages_curves"],
                ax=ax, hdi_prob=None
            )
            ax.set_xlim(0, 2)
            ax.set_title(f"{year[:4]} — {scenario_name}")

    fig.suptitle(f"Damage curves: {city}", size=16)
    plt.tight_layout()
    plt.show()

# Section 7: Return Year Bar Charts (Figure 12 style)

In [ ]:
# Horizontal bar chart: all 5 cities, 3 scenarios
# Focus on a single representative year (2026) for the policy note figure
FOCUS_YEAR = "2026"

fig, axes = plt.subplots(1, len(WALL_CITIES), figsize=(4 * len(WALL_CITIES), 5), sharey=True)

bar_height = 0.25
return_periods = [f"{ry} RY" for ry in RETURN_YEARS]
y_pos = np.arange(len(return_periods))

for ax, city in zip(axes, WALL_CITIES):
    city_year_df = ry_df.query('City == @city & Year == @FOCUS_YEAR')

    for i, (scenario_name, color) in enumerate(SCENARIO_COLORS.items()):
        scenario_data = city_year_df.query('Scenario == @scenario_name')
        damages = [scenario_data.query('`Return Period` == @rp')["Damage (USD millions)"].values[0]
                   for rp in return_periods]
        ax.barh(y_pos + i * bar_height, damages, bar_height,
                label=scenario_name if city == WALL_CITIES[0] else None,
                color=color, edgecolor='white', linewidth=0.5)

    ax.set_yticks(y_pos + bar_height)
    ax.set_yticklabels(return_periods, fontsize=10)
    ax.set_title(city, fontsize=12, fontweight='bold')
    ax.tick_params(axis='x', labelsize=9)
    for spine in ['bottom', 'top', 'right', 'left']:
        ax.spines[spine].set_visible(True)
        ax.spines[spine].set_color('lightgray')

axes[0].set_ylabel("Return Period", fontsize=12)
fig.supxlabel("Estimated Damages (2025 USD millions)", fontsize=12)
fig.legend(loc='upper center', ncol=3, fontsize=10, bbox_to_anchor=(0.5, 1.05), frameon=False)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'flood_wall_bar_chart_all_cities.png', format='png',
            bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Bar chart across multiple years for each city
years_to_chart = ["2026", "2030", "2050"]

for city in WALL_CITIES:
    fig, axes = plt.subplots(1, len(years_to_chart), figsize=(5 * len(years_to_chart), 4), sharey=True)

    for ax, yr in zip(axes, years_to_chart):
        city_year_df = ry_df.query('City == @city & Year == @yr')

        for i, (scenario_name, color) in enumerate(SCENARIO_COLORS.items()):
            scenario_data = city_year_df.query('Scenario == @scenario_name')
            damages = [scenario_data.query('`Return Period` == @rp')["Damage (USD millions)"].values[0]
                       for rp in return_periods]
            ax.barh(y_pos + i * bar_height, damages, bar_height,
                    label=scenario_name if ax == axes[0] else None,
                    color=color, edgecolor='white', linewidth=0.5)

        ax.set_yticks(y_pos + bar_height)
        ax.set_yticklabels(return_periods, fontsize=10)
        ax.set_title(yr, fontsize=12)
        ax.tick_params(axis='x', labelsize=9)

    axes[0].set_ylabel("Return Period", fontsize=12)
    fig.suptitle(f"Flood Wall Simulations: {city}", fontsize=14, fontweight='bold')
    fig.supxlabel("Estimated Damages (2025 USD millions)", fontsize=12)
    fig.legend(loc='upper center', ncol=3, fontsize=10, bbox_to_anchor=(0.5, 1.0), frameon=False)
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / f'flood_wall_bar_chart_{city.replace(" ", "_").lower()}.png',
                format='png', bbox_inches='tight', dpi=150)
    plt.show()

# Section 8: Summary Tables

In [ ]:
# Pivot table: damages by city and scenario for each return period (focus year)
for rp in return_periods:
    print(f"\n{'='*60}")
    print(f"Return Period: {rp} — Year: {FOCUS_YEAR}")
    print(f"{'='*60}")
    
    rp_data = ry_df.query('`Return Period` == @rp & Year == @FOCUS_YEAR')
    pivot = rp_data.pivot_table(
        index='City', columns='Scenario',
        values='Damage (USD millions)', aggfunc='first'
    )[list(SCENARIO_COLORS.keys())]
    
    display(pivot.round(2))

In [ ]:
# Damage reduction table (absolute and percentage)
reduction_rows = []
for _, row in ry_df.iterrows():
    if row["Scenario"] != "No wall":
        base_damage = ry_df.query(
            'City == @row.City & Year == @row.Year & `Return Period` == @row["Return Period"] & Scenario == "No wall"'
        )["Damage (USD millions)"].values[0]
        
        reduction = base_damage - row["Damage (USD millions)"]
        pct_reduction = (reduction / base_damage * 100) if base_damage > 0 else 0
        
        reduction_rows.append({
            "City": row["City"],
            "Scenario": row["Scenario"],
            "Year": row["Year"],
            "Return Period": row["Return Period"],
            "Damage Reduction (USD M)": reduction,
            "Reduction (%)": pct_reduction,
        })

reduction_df = pd.DataFrame(reduction_rows)

# Show reduction for focus year, 50 RY
print(f"Damage reduction at 50 RY — Year {FOCUS_YEAR}")
reduction_50ry = reduction_df.query('`Return Period` == "50 RY" & Year == @FOCUS_YEAR')
pivot_reduction = reduction_50ry.pivot_table(
    index='City', columns='Scenario',
    values=['Damage Reduction (USD M)', 'Reduction (%)'], aggfunc='first'
)
display(pivot_reduction.round(2))

In [ ]:
# Full summary table across all return periods for the focus year
print(f"\nFull damage reduction summary — Year {FOCUS_YEAR}")
full_summary = reduction_df.query('Year == @FOCUS_YEAR').pivot_table(
    index=['City', 'Return Period'],
    columns='Scenario',
    values='Damage Reduction (USD M)',
    aggfunc='first'
)
display(full_summary.round(2))